# `code/pipeline/p001_52_balance_maxt.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_52 — R5 E-1/E-2 + R5-4: 균형검정(P001-46)의 재통계 — max-|t| 결합 순열(벡터화) · **다중 라운드 셀 위의 비희석 균형** · 희석 항등식 · Table 10 A 희석 비중

[왜] (E-2) P001-46 의 결합 Mahalanobis 는 +단계 셀에서 퇴화했다(분산 0 공변량 → 준특이 공분산; p=0.978 은 증거가 아니다). (E-1) "6/10 균형" 은 비유의로
 균형을 인증한 것(rule 11 위반). (R5-4, 설계 심판) 회사 수준 공변량은 같은 라운드의 공동귀속 쌍 안에서 값이 같으므로 균형 계수도 Table 3 과 **같은
 희석 항등식** β_all = (1−d)·β_multi 를 따른다 — 원고의 "최대 표준화 차 0.17" 은 희석 추정량의 값이고, +단계 셀의 좁은 sd 단위 구간(±0.2 sd)은 비희석
 척도로 ±1.3 sd 라서 "균형" 이 아니라 "미식별" 이다. 따라서 (1) 결합 검정은 t 기반 max-|t| 로(t 는 희석에 불변), (2) 등가 서술은 **다중 라운드 셀**(식별 비교가
 실제로 있는 셀) 위의 비희석 계수·부트 구간·sd 단위로만, (3) 항등식과 d 를 공변량별로 보고, (4) 같은 항등식이 Table 10 A 의 무조건 행(2–3행: 결과가 라운드 내
 상수인 라운드 포함)에도 적용되므로 그 행들의 희석 비중을 낸다.

[구성] P001-46 과 동일한 공변량·모집단(sample_v1 NAEU FF, dt ≤ 2017-10; cell_cat / cell_stage; 전 딜 cell_stage 도). 셀 내 demean 회귀 fp 계수, 투자사 군집 강건 t
 (벡터화). 결합 = max_c |t_c| (9 공변량; prev_investor_count 커버 64% 는 따로) — 셀 내 fp 순열 1,000(공변량별 표본 안에서 lexsort 셔플), 크기 보정 100×100.
 다중 라운드 셀 = 파트너 행이 ≥2 개의 서로 다른 funding_round 에 속하는 혼합 셀(P001-49 정의). 비희석 균형: 그 셀 안의 계수·투자사 군집 부트 400·sd 단위 구간·MDE80(sd)
 ·max-|t| 결합. 희석 d_c = 단일 라운드 셀의 Σx̃² 비중(공변량 표본별). 항등식 검사 |β_all − (1−d)β_multi| < 1e-6.
 Table 10 A 희석: P001-42 표본(v6_common)에서 무조건 재참여(y1, 다음 라운드 없음 = 0)·무조건 재귀속(y2) 의 Σx̃² 중 결과 상수 라운드 비중.
[사전 예측] (2026-09-09, 결과 조회 전; 1차 실행(느린 판)의 FF cat 결과 — max|t| 2.24, p 0.194, 크기 0.080 — 는 알고 있음; 다중 라운드 판은 미지)
 결합(전 셀): FF cat p ∈ [0.10, 0.30] (1차 재현), 크기 보정 0.02–0.09; FF stage p > 0.30; ALL stage p ∈ [0.05, 0.60].
 비희석(다중 라운드, FF cat): 표준화 차 prev_investor_count −0.40~−0.60, n_founders −0.20~−0.35, n_female_founders +0.15~+0.30; MDE80(sd) 0.25–0.50; ±0.2 sd 등가 성립 공변량 ≤ 2/9.
 FF stage 다중 라운드 셀 ≤ 25 · 딜 ≤ 60 → 비희석 MDE80(sd) ≥ 0.6 → "미식별" 판정. 항등식 성립 11/11(전 셀 대 다중 라운드).
 Table 10 A: 무조건 y1 의 결과 상수 라운드 Σx̃² 비중 0.20–0.35; 무조건 y2 0.30–0.50.
[판정] 진단 — status OK. 원고 §4 ¶3 균형 문장은 비희석 판으로 다시 쓴다(E-1 미검출형 + 실제 크기), +단계 셀은 "식별 불가(셀 n)" 로.

[정정 2026-09-09] 1차 판(셀별 람다 순열)은 전 딜 패널에서 수 시간이 걸려 중단·벡터화 재작성; 다중 라운드 판·Table 10 A 희석은 설계 심판 R5-4 를 받아 추가.
```


In [ ]:
import json
import os

import numpy as np
import pandas as pd

from p001_v6_common import (CTX, CUT, EMP_BAND, RESCUE_SHA, V6_SHA, emit, equity_rounds, founders_by_org, investor_rows, load_sample, log, org_maps,
                            partner_gender_rows, qci)

rng = np.random.default_rng(20260952)
NPERM, NCAL, NPERM_CAL, NB = 1000, 100, 100, 400
OUT = {}
HERE = os.path.dirname(os.path.abspath(__file__))
P46 = json.load(open(os.path.join(os.environ.get("P001_ARTIFACTS", os.path.join(HERE, "..", "..", "artifacts")), "P00146.json"), encoding="utf-8"))["estimates"]
d = load_sample()
R = equity_rounds(set(d["org_uuid"]))
orgs = CTX.orgs.set_index("uuid")
fnd = founders_by_org()
ALLb = d[d["dt"] <= CUT].copy()
base = ALLb
base["founded"] = pd.to_datetime(base["org_uuid"].map(orgs["founded_on"]), errors="coerce")
base["age"] = (base["dt"] - base["founded"]).dt.days / 365.25
base["emp_band"] = base["org_uuid"].map(orgs["employee_count"]).map(EMP_BAND)
base["hq_us"] = (base["country_code"] == "USA").astype(float)
rr = R.set_index("uuid")
base["seq"] = base["funding_round_uuid"].map(rr["seq"])
base["ln_prior_rounds"] = np.log1p(base["seq"])
cum_amt = R.assign(a=R["amt"].fillna(0)).groupby("org_uuid")["a"].cumsum() - R["amt"].fillna(0)
rr_cum = pd.Series(cum_amt.to_numpy(), index=R["uuid"])
base["ln_prior_capital"] = np.log1p(base["funding_round_uuid"].map(rr_cum))
prev_ic = R.set_index("uuid")["prev_uuid"].map(rr["investor_count"]) if "investor_count" in rr.columns else None
base["prev_investor_count"] = pd.to_numeric(base["funding_round_uuid"].map(prev_ic), errors="coerce") if prev_ic is not None else np.nan
for c in ("n_founders", "n_female_founders", "founder_degree_share", "serial_share"):
    base[c] = base["org_uuid"].map(fnd[c])
base["ln_round_size"] = np.log1p(pd.to_numeric(base["funding_round_uuid"].map(rr["amt"]), errors="coerce"))
PRE9 = ["age", "ln_prior_rounds", "ln_prior_capital", "emp_band", "n_founders", "n_female_founders", "founder_degree_share", "serial_share", "hq_us"]
SEP = "prev_investor_count"; POST = "ln_round_size"
ALLb = base
FFb = ALLb[ALLb["ff"] == 1].copy()


class Frame:
    """공변량 표본 하나: 셀 id, 군집 id, 셀 내 demean 된 y, 셀 내 순열용 정렬 인덱스."""

    def __init__(self, dd, y, cell):
        self.n = len(dd); self.y = y
        self.cid = pd.factorize(dd[cell])[0]; self.gid = pd.factorize(dd["investor_uuid"])[0]
        self.cnt = np.bincount(self.cid).astype(float)
        yv = dd[y].to_numpy(float); self.yr = yv - (np.bincount(self.cid, yv) / self.cnt)[self.cid]
        self.fp = dd["fp"].to_numpy(float)
        self.base_order = np.lexsort((np.zeros(self.n), self.cid))
        self.sd = float(dd[y].std())
        self.rounds = dd["funding_round_uuid"].to_numpy()
        self.inv = dd["investor_uuid"].to_numpy()
        self.idx = dd.index.to_numpy()

    def xr(self, fp):
        return fp - (np.bincount(self.cid, fp) / self.cnt)[self.cid]

    def bt(self, fp):
        xr = self.xr(fp); sxx = float((xr * xr).sum())
        if sxx <= 0:
            return np.nan, np.nan, 0.0
        b = float((xr * self.yr).sum() / sxx); e = self.yr - b * xr
        s = np.bincount(self.gid, xr * e); se = float(np.sqrt((s * s).sum()) / sxx)
        return b, (b / se if se > 0 else np.nan), sxx

    def perm(self):
        shuf = np.lexsort((rng.random(self.n), self.cid)); out = np.empty_like(self.fp); out[self.base_order] = self.fp[shuf]; return out


def mixed_cells(df, cell):
    g = df.groupby(cell)["fp"].agg(["mean", "size"]); return g.index[(g["mean"] > 0) & (g["mean"] < 1)]


def joint_maxt(frames, fp_common, cid_common):
    """max-|t| 결합 순열 검정. [코드 검토 정정 2026-09-09] 귀무 추첨은 **공통 프레임에서 한 번** 셀 내 순열하고 각 공변량 표본으로 잘라 쓴다 — 공변량별 독립 순열은
    독립 |t| 들의 최댓값이라 귀무분포가 위로 치우쳐 p 가 과대(검토자 재현: 0.245 vs 0.172). frames[c].idx = 공통 프레임 안의 행 위치(프레임은 reset_index 됨)."""
    ks = [c for c, f in frames.items() if np.isfinite(f.bt(f.fp)[1])]
    tobs = {c: abs(frames[c].bt(frames[c].fp)[1]) for c in ks}; maxt = max(tobs.values())
    base_common = np.lexsort((np.zeros(len(fp_common)), cid_common))

    def perm(fp):
        shuf = np.lexsort((rng.random(len(fp)), cid_common)); out = np.empty_like(fp); out[base_common] = fp[shuf]; return out

    def draw(fpc):
        vals = [abs(frames[c].bt(fpc[frames[c].idx])[1]) for c in ks]; vals = [v for v in vals if np.isfinite(v)]; return max(vals) if vals else np.nan
    null = np.array([draw(perm(fp_common)) for _ in range(NPERM)])
    p = float(np.mean(null >= maxt)); rej = 0
    for _ in range(NCAL):
        fake = perm(fp_common); st = draw(fake)
        nl = np.array([draw(perm(fake)) for _ in range(NPERM_CAL)])   # 가짜 처치 아래 귀무 = fake 의 셀 내 재순열
        rej += int(np.mean(nl >= st) < 0.05)
    return {"covariates": ks, "max_abs_t": round(maxt, 3), "argmax": max(tobs, key=tobs.get), "p_perm": round(p, 4), "n_perm": NPERM, "null_p95": round(float(np.percentile(null, 95)), 3),
            "size_calibration_reject_rate": round(rej / NCAL, 3), "n_cal": NCAL, "n_perm_cal": NPERM_CAL, "permutation": "shared within-cell permutation on the common frame"}


def boot_multi(dd, y, cell, nb=NB):
    """다중 라운드 셀 표본에서 fp 계수의 투자사 군집 부트 (P001-49 boot_ci 와 동일 방식)."""
    fr = Frame(dd, y, cell); b = fr.bt(fr.fp)[0]
    grp = {c: g.index.to_numpy() for c, g in dd.groupby("investor_uuid")}; kl = list(grp); bs = []
    for _ in range(nb):
        pick = rng.integers(0, len(kl), len(kl)); s = dd.loc[np.concatenate([grp[kl[i]] for i in pick])]
        v = Frame(s, y, cell).bt(s["fp"].to_numpy(float))[0]
        if np.isfinite(v): bs.append(v)
    bs = np.array(bs); lo, hi = qci(bs); se = float(np.std(bs, ddof=1))
    return b, [lo, hi], se


def panel(df, cell, tag, p46_key):
    mixed = mixed_cells(df, cell); m = df[df[cell].isin(mixed)].copy().reset_index(drop=True)   # 위치 인덱스 = 공통 프레임 행 위치 (joint_maxt 슬라이싱용)
    nr = m.groupby(cell)["funding_round_uuid"].nunique(); single = set(nr.index[nr == 1]); multi = set(nr.index[nr >= 2])
    res = {"n_mixed_cells": int(len(mixed)), "n_deals": int(len(m)), "n_single_round_cells": len(single), "n_multi_round_cells": len(multi), "n_deals_multi": int(m[cell].isin(multi).sum()), "by_cov": {}}
    frames = {}
    for c in PRE9 + [SEP, POST]:
        dd = m.dropna(subset=[c])
        if len(dd) < 50 or dd[cell].nunique() < 10:
            res["by_cov"][c] = None; continue
        fr = Frame(dd, c, cell); b, t, sxx = fr.bt(fr.fp)
        xr2 = fr.xr(fr.fp) ** 2; is_single = dd[cell].isin(single).to_numpy(); dil = float(xr2[is_single].sum() / xr2.sum()) if xr2.sum() > 0 else np.nan
        p46 = (P46.get(p46_key, {}).get("by_cov", {}) or {}).get(c)
        # t_cluster_all_nodof: 자유도 보정 없음 — 순열 p 에만 쓴다(1.96 과 비교 금지)
        row = {"coef_all": round(b, 5), "t_cluster_all_nodof": round(t, 3) if np.isfinite(t) else None, "sd_all": round(fr.sd, 5), "std_diff_all": round(b / fr.sd, 4) if fr.sd > 0 else None,
               "ci95_boot_all_p46": p46["ci95"] if p46 else None, "sig_all_p46": bool(p46["sig"]) if p46 else None, "dilution_share_sxx_single_round": round(dil, 4), "n_all": int(len(dd))}
        mm = dd[dd[cell].isin(multi)]
        if len(mm) >= 30 and mm[cell].nunique() >= 8:
            bm, ci, se = boot_multi(mm, c, cell); sdm = float(mm[c].std())
            row.update({"coef_multi": round(bm, 5), "ci95_multi": [round(ci[0], 5), round(ci[1], 5)], "se_boot_multi": round(se, 5), "sd_multi": round(sdm, 5),
                        "std_diff_multi": round(bm / sdm, 4) if sdm > 0 else None, "ci95_multi_sd_units": [round(ci[0] / sdm, 4), round(ci[1] / sdm, 4)] if sdm > 0 else None,
                        "mde80_multi_sd": round(2.8 * se / sdm, 4) if sdm > 0 else None, "sig_multi": bool(ci[0] > 0 or ci[1] < 0),
                        "within_pm0.2sd_multi": bool(sdm > 0 and ci[0] / sdm >= -0.2 and ci[1] / sdm <= 0.2), "n_multi": int(len(mm)), "n_cells_multi": int(mm[cell].nunique()),
                        "identity_abs_err": round(abs(b - (1 - dil) * bm), 8)})
            log(f"  {tag:<9} {c:<22} all β {b:+.4f} t {t:+.2f} d(sd) {row['std_diff_all']} · dil {dil:.3f} · multi β {bm:+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}] d(sd) {row['std_diff_multi']} CI(sd) {row['ci95_multi_sd_units']} "
                f"MDE(sd) {row['mde80_multi_sd']} ±0.2 {row['within_pm0.2sd_multi']} n {len(mm)}/{row['n_cells_multi']} · 항등식 오차 {row['identity_abs_err']:.1e}")
        else:
            row.update({"coef_multi": None, "n_multi": int(len(mm)), "n_cells_multi": int(mm[cell].nunique()) if len(mm) else 0})
            log(f"  {tag:<9} {c:<22} all β {b:+.4f} t {t:+.2f} · dil {dil:.3f} · multi: 식별 불가 (n {len(mm)}, 셀 {row['n_cells_multi']})")
        res["by_cov"][c] = row
        if c in PRE9: frames[c] = fr
    res["joint_maxt_all"] = joint_maxt(frames, m["fp"].to_numpy(float), pd.factorize(m[cell])[0])
    j = res["joint_maxt_all"]
    log(f"  {tag:<9} 결합(전 셀) max|t| {j['max_abs_t']:.2f} ({j['argmax']}) · p {j['p_perm']:.3f} · 귀무 p95 {j['null_p95']:.2f} · 크기 보정 {j['size_calibration_reject_rate']:.3f}")
    mm_all = m[m[cell].isin(multi)].reset_index(drop=True)
    frames_m = {c: Frame(mm_all.dropna(subset=[c]), c, cell) for c in PRE9 if len(mm_all.dropna(subset=[c])) >= 30 and mm_all.dropna(subset=[c])[cell].nunique() >= 8}
    if len(frames_m) >= 3:
        res["joint_maxt_multi"] = joint_maxt(frames_m, mm_all["fp"].to_numpy(float), pd.factorize(mm_all[cell])[0]); jm = res["joint_maxt_multi"]
        log(f"  {tag:<9} 결합(다중 라운드) max|t| {jm['max_abs_t']:.2f} ({jm['argmax']}) · p {jm['p_perm']:.3f} · 크기 보정 {jm['size_calibration_reject_rate']:.3f}")
    else:
        res["joint_maxt_multi"] = None
    ks = [c for c in PRE9 if res["by_cov"].get(c) and res["by_cov"][c].get("coef_multi") is not None]
    res["equivalence_multi"] = {"n_cov": len(ks), "within_pm0.2sd": int(sum(res["by_cov"][c]["within_pm0.2sd_multi"] for c in ks)),
                                "n_sig_multi": int(sum(res["by_cov"][c]["sig_multi"] for c in ks)), "max_abs_std_diff_multi": round(max((abs(res["by_cov"][c]["std_diff_multi"]) for c in ks), default=float("nan")), 4),
                                "argmax_abs_std_diff_multi": max(ks, key=lambda c: abs(res["by_cov"][c]["std_diff_multi"])) if ks else None,
                                "mde80_sd_range": [round(min(res["by_cov"][c]["mde80_multi_sd"] for c in ks), 3), round(max(res["by_cov"][c]["mde80_multi_sd"] for c in ks), 3)] if ks else None}
    res["identity_max_abs_err"] = max((res["by_cov"][c]["identity_abs_err"] for c in res["by_cov"] if res["by_cov"][c] and res["by_cov"][c].get("identity_abs_err") is not None), default=None)
    return res


log("\n" + "=" * 100 + "\n[FF cat] 회사×연×섹터\n" + "=" * 100)
OUT["FF_cell_cat_pre"] = panel(FFb, "cell_cat", "FF cat", "FF_cell_cat_pre")
log("\n" + "=" * 100 + "\n[FF stage] +단계\n" + "=" * 100)
OUT["FF_cell_stage_pre"] = panel(FFb, "cell_stage", "FF stage", "FF_cell_stage_pre")
log("\n" + "=" * 100 + "\n[ALL stage] 전 딜 +단계\n" + "=" * 100)
OUT["ALL_cell_stage_pre"] = panel(ALLb, "cell_stage", "ALL stage", "ALL_cell_stage_pre")


In [ ]:
# ── Table 10 A 무조건 행의 희석 (P001-42 표본 재구성) ──────────────────────────
log("\n" + "=" * 100 + "\n[Table 10 A] 무조건 행의 결과 상수 라운드 Σx̃² 비중\n" + "=" * 100)
om = org_maps(d); Rq = equity_rounds(set(d["org_uuid"]))
W0, W1 = pd.Timestamp("2010-01-01"), pd.Timestamp("2020-10-31")
R0 = Rq[(Rq["rdt"] >= W0) & (Rq["rdt"] <= W1)].copy(); R0["ff"] = R0["org_uuid"].map(om["ff"])
R0["next36"] = ((R0["next_dt"] - R0["rdt"]).dt.days <= 1095).fillna(False).astype(float)
I = investor_rows(R0["uuid"]); pt = partner_gender_rows(R0["uuid"])
pa = pt.groupby(["funding_round_uuid", "investor_uuid"]).agg(fp=("fp", "max"), partners=("partner_uuid", lambda s: frozenset(s))).reset_index()
X = I.merge(pa, on=["funding_round_uuid", "investor_uuid"], how="inner").merge(R0[["uuid", "org_uuid", "ff", "next_uuid", "next36"]], left_on="funding_round_uuid", right_on="uuid")
inv_all = investor_rows(set(R0["next_uuid"].dropna())); next_inv = inv_all.groupby("funding_round_uuid")["investor_uuid"].agg(set).to_dict()
pt_next = partner_gender_rows(set(R0["next_uuid"].dropna())); next_part = pt_next.groupby(["funding_round_uuid", "investor_uuid"])["partner_uuid"].agg(frozenset).to_dict()
X["y1"] = [1.0 if (isinstance(nu, str) and inv in next_inv.get(nu, set())) else 0.0 for nu, inv in zip(X["next_uuid"], X["investor_uuid"])]
X["y2"] = [1.0 if (isinstance(nu, str) and len(next_part.get((nu, inv), frozenset()) & ps) > 0) else 0.0 for nu, inv, ps in zip(X["next_uuid"], X["investor_uuid"], X["partners"])]
g = X.groupby("funding_round_uuid")["fp"].agg(["mean", "size"]); mixed = g.index[(g["mean"] > 0) & (g["mean"] < 1) & (g["size"] >= 2)]
M = X[X["funding_round_uuid"].isin(mixed)].copy(); MF = M[M["ff"] == 1]; Mc = M[M["next36"] == 1]; McF = Mc[Mc["ff"] == 1]


def dil_rounds(df, y):
    xr2 = (df["fp"] - df.groupby("funding_round_uuid")["fp"].transform("mean")) ** 2
    yv = df.groupby("funding_round_uuid")[y].transform(lambda s: s.var(ddof=0)); const = (yv == 0)
    return {"n_rounds": int(df["funding_round_uuid"].nunique()), "n_rounds_outcome_constant": int(df.loc[const, "funding_round_uuid"].nunique()),
            "share_sxx_outcome_constant_rounds": round(float(xr2[const].sum() / xr2.sum()), 4) if xr2.sum() > 0 else None}


T10 = {"unconditional_y1_FF": dil_rounds(MF, "y1"), "unconditional_y2_FF": dil_rounds(MF, "y2"), "conditional_y1_FF": dil_rounds(McF, "y1"), "conditional_y2_FF": dil_rounds(McF, "y2"),
       "n_ff_mixed_rounds_no_next36": int(MF.loc[MF["next36"] == 0, "funding_round_uuid"].nunique())}
for k, v in T10.items():
    log(f"  {k}: {v}")
OUT["Table10A_dilution"] = T10


In [ ]:
# ── 판정 ────────────────────────────────────────────────────────────────────
ffc, ffs, alls = OUT["FF_cell_cat_pre"], OUT["FF_cell_stage_pre"], OUT["ALL_cell_stage_pre"]
bc = ffc["by_cov"]; qm = ffc["equivalence_multi"]
def sdm(c):
    return bc[c]["std_diff_multi"] if bc.get(c) and bc[c].get("std_diff_multi") is not None else float("nan")
pred = {"FFcat_all_p_in_[0.10,0.30]": 0.10 <= ffc["joint_maxt_all"]["p_perm"] <= 0.30, "FFcat_size_in_[0.02,0.09]": 0.02 <= ffc["joint_maxt_all"]["size_calibration_reject_rate"] <= 0.09,
        "FFstage_all_p_gt_0.30": ffs["joint_maxt_all"]["p_perm"] > 0.30, "ALLstage_p_in_[0.05,0.60]": 0.05 <= alls["joint_maxt_all"]["p_perm"] <= 0.60,
        "multi_prev_ic_in_[-0.60,-0.40]": -0.60 <= sdm(SEP) <= -0.40, "multi_n_founders_in_[-0.35,-0.20]": -0.35 <= sdm("n_founders") <= -0.20,
        "multi_n_ff_in_[0.15,0.30]": 0.15 <= sdm("n_female_founders") <= 0.30, "multi_mde_sd_in_[0.25,0.50]": bool(qm["mde80_sd_range"] and 0.25 <= qm["mde80_sd_range"][0] and qm["mde80_sd_range"][1] <= 0.50),
        "multi_equiv_0.2sd_le2": qm["within_pm0.2sd"] <= 2, "FFstage_multi_unidentified": ffs["n_multi_round_cells"] <= 25 and ffs["n_deals_multi"] <= 60,
        "identity_holds": bool(ffc["identity_max_abs_err"] is not None and ffc["identity_max_abs_err"] < 1e-6),
        "T10A_y1u_share_0.20_0.35": bool(T10["unconditional_y1_FF"]["share_sxx_outcome_constant_rounds"] is not None and 0.20 <= T10["unconditional_y1_FF"]["share_sxx_outcome_constant_rounds"] <= 0.35),
        "T10A_y2u_share_0.30_0.50": bool(T10["unconditional_y2_FF"]["share_sxx_outcome_constant_rounds"] is not None and 0.30 <= T10["unconditional_y2_FF"]["share_sxx_outcome_constant_rounds"] <= 0.50)}
pred = {k: bool(v) for k, v in pred.items()}
OUT["prediction_check"] = pred
verdict = (f"FF cat 결합(전 셀) max|t| {ffc['joint_maxt_all']['max_abs_t']:.2f} p {ffc['joint_maxt_all']['p_perm']:.3f} 크기 {ffc['joint_maxt_all']['size_calibration_reject_rate']:.3f} | 다중 라운드({ffc['n_multi_round_cells']} 셀/{ffc['n_deals_multi']} 딜) "
           f"최대 |d| {qm['max_abs_std_diff_multi']} ({qm['argmax_abs_std_diff_multi']}), 유의 {qm['n_sig_multi']}/{qm['n_cov']}, ±0.2sd 등가 {qm['within_pm0.2sd']}/{qm['n_cov']}, MDE(sd) {qm['mde80_sd_range']}; "
           f"결합(다중) p {ffc['joint_maxt_multi']['p_perm'] if ffc.get('joint_maxt_multi') else 'NA'} | FF stage 다중 {ffs['n_multi_round_cells']} 셀/{ffs['n_deals_multi']} 딜 → {'식별 불가' if ffs['equivalence_multi']['n_cov'] == 0 else ffs['equivalence_multi']} · 결합(전 셀) p {ffs['joint_maxt_all']['p_perm']:.3f} | "
           f"ALL stage p {alls['joint_maxt_all']['p_perm']:.3f} | 항등식 최대 오차 {ffc['identity_max_abs_err']} | Table 10 A 무조건 y1 결과 상수 라운드 Σx̃² 비중 {T10['unconditional_y1_FF']['share_sxx_outcome_constant_rounds']} · y2 {T10['unconditional_y2_FF']['share_sxx_outcome_constant_rounds']} (예측 적중 {sum(pred.values())}/{len(pred)})")
emit("P001-52", "R5 E-1/E-2 + R5-4: 균형검정 재통계 — max-|t| 결합 순열 · 다중 라운드 셀 위 비희석 균형(sd 단위·MDE) · 희석 항등식 · Table 10 A 희석", "OK", OUT,
     prediction="FF cat p∈[0.10,0.30] 크기 0.02–0.09; 다중 라운드 prev_ic d −0.40~−0.60 · n_founders −0.20~−0.35 · n_ff +0.15~+0.30 · MDE(sd) 0.25–0.50 · ±0.2sd ≤2/9; FF stage 다중 ≤25 셀(미식별); 항등식 성립; T10A y1u 0.20–0.35",
     verdict=verdict, kill_met=False, n=int(ffc["n_deals"]),
     extra={"stage": 7, "feeds": "R5 ident E-1/E-2 · design R5-4 → §4 ¶3 균형 문장 · Table 10 B · Table 10 A 각주", "slug": "balance_maxt", "builds_on": "P001-46/49/42",
            "common_sha256_16": RESCUE_SHA, "v6_common_sha256_16": V6_SHA})
log("done")
